# Import Packages


In [1]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

True

# Implement response function


In [2]:
def get_chatbot_response(client, messages, temperature=0.0):
    
    input_messages = []
    
    for message in messages:
        input_messages.append({"role": message["role"], "content": message["content"]})
        
    response = client.chat.completions.create(
        model=os.getenv("MODEL_NAME"),
        messages = input_messages,
        max_tokens = 2000,
        temperature = temperature,
        top_p=0.8,
    )
    
    return response.choices[0].message.content
    

In [3]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

## Get LLM Response


In [4]:
messages = [{'role':'user','content':"What's the capital of Sri Lanka?"}]
response = get_chatbot_response(client,messages)
print(response)

The capital of Sri Lanka is Colombo.


# Prompt Engineering


In [5]:
system_prompt = """
You are a helpful assistant that answer questions about capitals of countries.

Your output should be in a structured json format exactly like the one bellow. You are not allowed to write anything other than the json object:
[
{
    country: the country that you will get the capital of 
    capital: the capital of the country stated
}
]
"""
messages = [{'role':'system','content':system_prompt}]
messages.append({'role':'user','content':"What's the capital of Italy?"})

In [6]:
response = get_chatbot_response(client, messages)
print(response)

[
    {
        "country": "Italy",
        "capital": "Rome"
    }
]


In [7]:
response = json.loads(response)

## Structuring Inputs


In [8]:
user_input = """
    Get me the capitals of the following countries:
    - India
    - Malaysia
    - Germany
    - Pakistan
"""

In [9]:
messages = [
    {
        'role':'system',
        'content': system_prompt
    },
    {
        'role':'user',
        'content': user_input
    }
]

In [10]:
response = json.loads(get_chatbot_response(client, messages))

In [11]:
response

[{'country': 'India', 'capital': 'New Delhi'},
 {'country': 'Malaysia', 'capital': 'Kuala Lumpur'},
 {'country': 'Germany', 'capital': 'Berlin'},
 {'country': 'Pakistan', 'capital': 'Islamabad'}]

## Model Thinking Patterns


In [12]:
user_prompt = """
Calculate the result of this equation: 1+3

Your output should be in a structured json format exactly like the one bellow. You are not allowed to write anything other than the json object:
{
    result: The final number resulted from calculating the equation above
}
"""

In [13]:
messages = [
    {
        'role': 'user',
        'content': user_prompt
    }
]

In [14]:
response = json.loads(get_chatbot_response(client, messages))
response

{'result': 4}

In [15]:
user_prompt = """
Calculate the result of this equation: 259/2*8654+91072*33-12971

Your output should be in a structured json format exactly like the one bellow. You are not allowed to write anything other than the json object:
{
    steps: This is where you solve the equation bit by bit following the BEDMAS order of operations. You need to show your work and calculate each step leading to final result. Feel free to write here in free text. 
    result: The final number resulted from calculating the equation above
}
"""

In [16]:
messages = [
    {
        'role': 'user',
        'content': user_prompt
    }
]

In [17]:
response = get_chatbot_response(client, messages)
print(response)

{
    "steps": "First, we calculate 259 divided by 2 which equals 129.5. Then we multiply 129.5 by 8654 to get 1120733. Next, we multiply 91072 by 33 to get 3008496. Finally, we subtract 12971 from the sum of the previous two results (1120733 + 3008496) to get the final result of 4130258.",
    "result": 4130258
}


# RAG - Retrieval Augmented Generation


In [18]:
user_prompt = """
    What is new in iphone 16?
"""

messages = [
    {
        'role': 'user',
        'content': user_prompt
    }
]

In [19]:
response = get_chatbot_response(client, messages)
print(response)

As of my last update, there is no information available about an iPhone 16. Apple typically releases new iPhone models on an annual basis, so it is possible that an iPhone 16 may be released in the future with new features and improvements. However, without official announcements from Apple, it is difficult to say what specific new features or changes may be included in an iPhone 16.


### Giving Context to the model


In [20]:
iphone_16 = """
The iPhone 16 introduces several exciting updates, making it one of Apple's most advanced smartphones to date. It features a larger 6.1-inch display for the base model and a 6.7-inch screen for the iPhone 16 Plus, with thinner bezels and a more durable Ceramic Shield. The iPhone 16 Pro and Pro Max boast even larger displays, measuring 6.3 and 6.9 inches respectively, offering the thinnest bezels seen on any Apple product so far.

Powered by the new A18 chip (A18 Pro for the Pro models), these phones deliver significant performance improvements, with enhanced neural engine capabilities, faster GPU for gaming, and machine learning tasks. The camera systems are also upgraded, with the base iPhone 16 sporting a dual-camera setup with a 48MP main sensor. The Pro models offer a 48MP Ultra Wide and 5x telephoto camera, enhanced by Apple’s "Camera Control" button for more flexible photography options.

Apple also introduced advanced audio features like "Audio Mix," which uses machine learning to separate background sounds from speech, allowing for more refined audio capture during video recording. Battery life has been extended, especially in the iPhone 16 Pro Max, which is claimed to have the longest-lasting battery of any iPhone 
9TO5MAC

APPLEMAGAZINE
.

Additionally, Apple has switched to USB-C for faster charging and data transfer, and the Pro models now support up to 2x faster video encoding. The starting prices remain consistent with previous generations, with the iPhone 16 starting at $799, while the Pro models start at $999
"""

user_prompt = f"""
    {iphone_16}
    What is new in iphone 16?
"""

In [21]:
messages = [
    {
        'role': 'user',
        'content': user_prompt
    }
]

response = get_chatbot_response(client, messages)
print(response)

Some of the new features in the iPhone 16 include a larger display, a more durable Ceramic Shield, the new A18 chip for improved performance, upgraded camera systems, advanced audio features like "Audio Mix," extended battery life, USB-C for faster charging and data transfer, and faster video encoding capabilities in the Pro models.


#### Automatically extract context data from DB


In [22]:
samsung_s23 = """
The Samsung Galaxy S23 brings some incremental but notable upgrades to its predecessor, the Galaxy S22. It features the Snapdragon 8 Gen 2 processor, a powerful chip optimized for the S23 series, delivering enhanced performance, especially for gaming and multitasking. This chip ensures top-tier speed and efficiency across all models, from the base S23 to the larger S23+ and S23 Ultra​
STUFF

TECHRADAR
.

In terms of design, the S23's camera module has been streamlined by removing the raised metal contour around the cameras, creating a cleaner, sleeker look. It also sports the same 6.1-inch 120Hz AMOLED display, protected by tougher Gorilla Glass Victus 2, making it more resistant to scratches and drops​
TECHRADAR
.

The S23 Ultra stands out with its 200MP main camera, offering impressive photo clarity, especially in low-light conditions. The selfie camera across the series has been updated to a 12MP sensor, resulting in sharper selfies. The Ultra model also includes productivity tools such as the S-Pen, which remains an essential feature for note-taking and creative tasks​
STUFF

TECHRADAR
.

Battery life is solid, with the S23 Ultra featuring a 5000mAh battery that lasts comfortably through a day of heavy use. However, charging speeds still lag behind some competitors, with 45W wired charging, which is slower than other brands offering up to 125W charging​
STUFF
.

Overall, the Galaxy S23 series enhances performance, durability, and camera quality, making it a strong contender for users seeking a high-performance flagship.
"""

In [23]:
data = [iphone_16, samsung_s23]

In [30]:
user_prompt = """
    What is new in samsung s23?
"""

In [25]:
def get_embeddings(client, text):
    output = client.embeddings.create(
        input=text,
        model=os.getenv("OPENAI_EMBEDDING_MODEL_NAME")
    )
    
    embeddings = []
    
    for i in output.data:
        embeddings.append(i.embedding)
    
    return embeddings

In [26]:
data_embeddings = [get_embeddings(client, text)[0] for text in data]

In [31]:
user_embeddings = get_embeddings(client, user_prompt)[0]

In [28]:
len(data_embeddings)

2

In [32]:
from sklearn.metrics.pairwise import cosine_similarity

data_similarity = cosine_similarity([user_embeddings], data_embeddings)
data_similarity

array([[0.35031066, 0.64922654]])

#### Which one has closest


In [34]:
closest_index = data_similarity.argmax()

In [35]:
data[closest_index]

"\nThe Samsung Galaxy S23 brings some incremental but notable upgrades to its predecessor, the Galaxy S22. It features the Snapdragon 8 Gen 2 processor, a powerful chip optimized for the S23 series, delivering enhanced performance, especially for gaming and multitasking. This chip ensures top-tier speed and efficiency across all models, from the base S23 to the larger S23+ and S23 Ultra\u200b\nSTUFF\n\nTECHRADAR\n.\n\nIn terms of design, the S23's camera module has been streamlined by removing the raised metal contour around the cameras, creating a cleaner, sleeker look. It also sports the same 6.1-inch 120Hz AMOLED display, protected by tougher Gorilla Glass Victus 2, making it more resistant to scratches and drops\u200b\nTECHRADAR\n.\n\nThe S23 Ultra stands out with its 200MP main camera, offering impressive photo clarity, especially in low-light conditions. The selfie camera across the series has been updated to a 12MP sensor, resulting in sharper selfies. The Ultra model also inclu

In [36]:
user_prompt_with_data = f"""
    {data[closest_index]}
 
    {user_prompt}
"""

In [37]:
messages = [
    {
        'role': 'user',
        'content': user_prompt_with_data
    }
]

response = get_chatbot_response(client, messages)

In [38]:
response

'Some of the new features in the Samsung Galaxy S23 include the Snapdragon 8 Gen 2 processor, a streamlined camera module design, tougher Gorilla Glass Victus 2 protection, a 200MP main camera on the Ultra model, updated selfie camera, and the inclusion of productivity tools like the S-Pen on the Ultra model.'